In [1]:
from MultiAgent_DQ import *
import numpy as np
from utils import *

In [ ]:
algo = Algorithm(e=0.1, lambda_val=0, 
                 alpha=0.1, delta=0.9,
                 random_uniform=True, tolerance=1e-5, 
                 M=100, verbose=False)
q1_history_per_sample, q2_history_per_sample = algo.run()


In [15]:
dfs_1  = convert_dict_to_mean_df(q1_history_per_sample, mean= False)
dfs_2  = convert_dict_to_mean_df(q2_history_per_sample, mean= False)

In [16]:
strategy_played_1 = {}
strategy_played_2 = {}

# Iterate over the dataframe rows
for i in range(len(dfs_1)):
    last_1 = dfs_1[i].iloc[-1]  # Get the last row using iloc (optimized)
    last_2 = dfs_2[i].iloc[-1]  # Same for second dataframe

    # Define conditions for both players
    GT_1 = (last_1['Q1'] > last_1['Q3']) & (last_1['Q2'] < last_1['Q4'])
    AD_1 = (last_1['Q1'] < last_1['Q3']) & (last_1['Q2'] < last_1['Q4'])
    AC_1 = (last_1['Q1'] > last_1['Q3']) & (last_1['Q2'] > last_1['Q4'])

    GT_2 = (last_2['Q1'] > last_2['Q3']) & (last_2['Q2'] < last_2['Q4'])
    AD_2 = (last_2['Q1'] < last_2['Q3']) & (last_2['Q2'] < last_2['Q4'])
    AC_2 = (last_2['Q1'] > last_2['Q3']) & (last_2['Q2'] > last_2['Q4'])

    # Assign strategy for both players
    strategy_map = {
        AD_1: 1,  # Always Defecting
        GT_1: 2,  # Grim Trigger
        AC_1: 3,  # Always Cooperating
    }

    # Handle strategy for player 1 and player 2
    for player, strategy in [(AD_1, strategy_map), (GT_1, strategy_map), (AC_1, strategy_map)]:
        if player:
            strategy_played_1[i] = strategy[player]
            break  # Once a strategy is found, break the loop to avoid multiple assignments

    for player, strategy in [(AD_2, strategy_map), (GT_2, strategy_map), (AC_2, strategy_map)]:
        if player:
            strategy_played_2[i] = strategy[player]
            break  # Once a strategy is found, break the loop to avoid multiple assignments




In [21]:
# Convert strategy_played_1 and strategy_played_2 dictionaries to DataFrames
df_strategy_1 = pd.DataFrame(list(strategy_played_1.items()), columns=['Index', 'Strategy_1'])
df_strategy_2 = pd.DataFrame(list(strategy_played_2.items()), columns=['Index', 'Strategy_2'])

# Merge the two DataFrames on 'Index'
df_merged = pd.merge(df_strategy_1, df_strategy_2, on='Index')

# Optionally, set 'Index' as the DataFrame index (if you need it)
df_merged.set_index('Index', inplace=True)

In [23]:
# Count how many times Strategy_1 and Strategy_2 are different
count_diff = (df_merged['Strategy_1'] != df_merged['Strategy_2']).sum()

# Display the result
print(f"Number of times Strategy_1 and Strategy_2 are different: {count_diff}")


Number of times Strategy_1 and Strategy_2 are different: 0


In [ ]:
def analyze_critical_delta_empirically(delta_values, epsilon_values):
    """
    Analyzes the critical value of delta for which the agent begins cooperating against the Grim Trigger strategy.

    Parameters:
        None.

    Returns:
        A plot of cooperation ratio vs. delta.
    """
    import numpy as np
    import matplotlib.pyplot as plt

    # Define the payoffs
    P, R, T, S = 0.2, 0.8, 1, 0

    # Create a 2D grid for strategies
    strategy_grid = np.zeros((len(delta_values), len(epsilon_values)))

    # Iterate over all (delta, epsilon) pairs
    for i, delta in enumerate(delta_values):
        for j, epsilon in enumerate(epsilon_values):
            if epsilon == 1:  # Avoid division by zero when epsilon is 1
                strategy_grid[i, j] = 0
                continue
            
            # Example calculation (replace this with actual logic)
            
            calculator = Theoretical_QValueCalculator(epsilon=epsilon, P=P, R=R, T=T, S=S, delta=delta)
            c1, c2 = calculator.calculate_c1_c2()

            k = 500 #if delta > c2 else 5 

            #Here in the comment,  I was incrementing only for AC because we had noticed that more iterations are needed in this case. 
            #However, it seems that the pairs of epsilon and delta values that lie at the boundaries between strategies still require more iterations. 

           
            strategy = epsilon_GrimTrigger()
            solver = PrisonersDilemmaSolver(
                strategy=strategy,
                e_initial=epsilon,
                lambda_val=0,
                alpha=0.1,
                delta=delta,
                K=k,
                M=1,
                e_decay=1,
                random_uniform=True,
                verbose=False)
            val_inner_value_iteration = solver.check_value_iteration()
            dfs = convert_dict_to_mean_df(val_inner_value_iteration, mean=False)
            last_1 = dfs[0][-1:]
            GT = np.any((last_1['Q1'] > last_1['Q3']) & (last_1['Q2'] < last_1['Q4']))
            AD = np.any((last_1['Q1'] < last_1['Q3']) & (last_1['Q2'] < last_1['Q4']))
            AC = np.any((last_1['Q1'] > last_1['Q3']) & (last_1['Q2'] > last_1['Q4']))
            
            if AD:
                strategy_grid[i, j] =  1  # e-Always Defecting
            elif GT:
                strategy_grid[i, j] =  2  # e-GrimTrigger
            elif AC:
                strategy_grid[i, j] = 3

            else:
                print('for delta value {delta} and epsilon value {epsilon} the value iteration is not converging correctly to a predefined strategy')
                strategy_grid[i, j] = 4

    # Debug: Ensure all values are in the expected range
    strategy_grid = np.clip(strategy_grid, 1, 4)  # Clip values to ensure they are in [1, 4]
    return strategy_grid

